# W12: Logistic Regression and Discrete Response Models

## Welcome to Module 12: Discrete Response Models

You have mastered the mechanics of Maximum Likelihood Estimation (MLE) and its application to continuous data. Now, we are shifting our focus to Discrete Response Models, specifically Logistic Regression.

This is a pivotal transition because, in your work as a pharmaceutical analyst, the most critical business questions are often binary:
- Will a patient be readmitted?
- Does this physician adopt the new treatment protocol?
- Did the marketing campaign convert to a sale?

By the end of this module, you will be able to handle yes/no data with the same rigor you apply to continuous forecasting, informing high-stakes business decisions.

In [ ]:
# Setup and Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.optimize import minimize
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc

# Set display options for pandas
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)

# Set random seed for reproducibility across the notebook
np.random.seed(42)

# Set seaborn style for better visualizations
sns.set_theme(style='whitegrid')
print("Libraries successfully imported and random seed set to 42.")

## 1. The Challenge: Why Linear Regression Fails for Binary Data

You cannot simply use Multiple Linear Regression for binary outcomes (0 or 1). If you do, you encounter the Linear Probability Model (LPM) problem:

- The Boundary Problem: A linear model can predict probabilities outside the logical [0, 1] range (e.g., predicting a probability of -0.2 or 1.5).
- Heteroskedasticity: The variance of the error terms in a binary model is not constant, which violates the fundamental assumptions of standard OLS regression.

Let's generate some synthetic pharmaceutical data to demonstrate this limitation. We will simulate Patient Age vs. Readmission Status (0 = Not Readmitted, 1 = Readmitted).

In [ ]:
# 1. Data Creation: Synthetic 1D Binary Data
# Simulate 150 patients
n_samples = 150

# Generate ages between 30 and 90
age = np.random.uniform(30, 90, n_samples)

# Create a true underlying relationship (log-odds)
# Older patients have a higher chance of readmission
true_z = -7.5 + 0.12 * age

# Convert to probabilities using the logistic function (hidden from our model for now)
true_prob = 1 / (1 + np.exp(-true_z))

# Generate binary outcomes based on the probabilities
readmitted = np.random.binomial(1, true_prob)

# Create DataFrame
df_pharma = pd.DataFrame({'Age': age, 'Readmitted': readmitted})

print("First 5 rows of our Patient Dataset:")
print(df_pharma.head())
print("\nDataset Info:")
print(df_pharma.info())

### Visualizing the Linear Probability Model (LPM) Failure

Let's fit a standard Ordinary Least Squares (OLS) Linear Regression model to this binary data and observe what happens.

In [ ]:
# Fit OLS Linear Regression
ols_model = LinearRegression()
X_age = df_pharma[['Age']].values
y_readmit = df_pharma['Readmitted'].values
ols_model.fit(X_age, y_readmit)

# Generate predictions across the age range
age_range = np.linspace(20, 100, 300).reshape(-1, 1)
ols_predictions = ols_model.predict(age_range)

# Visualization
plt.figure(figsize=(10, 6))
plt.scatter(df_pharma['Age'], df_pharma['Readmitted'], alpha=0.5, color='blue', label='Actual Patient Data')
plt.plot(age_range, ols_predictions, color='red', linewidth=2, label='OLS Linear Fit (LPM)')

# Highlight boundaries
plt.axhline(1, color='gray', linestyle='--', alpha=0.7)
plt.axhline(0, color='gray', linestyle='--', alpha=0.7)

# Highlight regions where predictions are illogical
plt.fill_between(age_range.flatten(), 1, ols_predictions.flatten(), 
                 where=(ols_predictions.flatten() > 1), color='red', alpha=0.2, label='Prob > 1')
plt.fill_between(age_range.flatten(), 0, ols_predictions.flatten(), 
                 where=(ols_predictions.flatten() < 0), color='red', alpha=0.2, label='Prob < 0')

plt.title('The Failure of Linear Regression on Binary Data', fontsize=14)
plt.xlabel('Patient Age', fontsize=12)
plt.ylabel('Probability of Readmission', fontsize=12)
plt.legend(loc='upper left')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("Notice how the red line extends below 0 and above 1. A probability of -0.1 or 1.2 is nonsensical!")

## 2. Core Concept 1: The Sigmoid (Logit) Function

Instead of modeling the probability directly as a straight line, we use the Logit Link Function. This uses the Logistic Distribution to squish any input (from negative infinity to positive infinity) into a value strictly between 0 and 1.

The Sigmoid function formula is:
p = 1 / (1 + exp(-z))

Where z is our standard linear combination: z = beta_0 + beta_1 * X

In [ ]:
def sigmoid(z):
    """Compute the sigmoid function."""
    return 1 / (1 + np.exp(-z))

# Let's visualize the Sigmoid function
z_values = np.linspace(-10, 10, 200)
sigmoid_values = sigmoid(z_values)

plt.figure(figsize=(8, 5))
plt.plot(z_values, sigmoid_values, color='green', linewidth=3, label='Sigmoid Function')
plt.axvline(0, color='gray', linestyle='--', alpha=0.7)
plt.axhline(0.5, color='gray', linestyle='--', alpha=0.7, label='Decision Threshold (p=0.5)')
plt.title('The Sigmoid Function: Squashing inputs to [0, 1]', fontsize=14)
plt.xlabel('z (Linear Combination: beta_0 + beta_1*X)', fontsize=12)
plt.ylabel('Predicted Probability (p)', fontsize=12)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Core Concept 2: Odds and Log-Odds

How do we connect our linear equation (z) to this probability (p)? Through Odds and Log-Odds.

- Probability (p): The chance of an event occurring (e.g., 0.8 or 80%).
- Odds: The ratio of success to failure: p / (1 - p). If p=0.8, odds = 0.8 / 0.2 = 4 (or 4 to 1).
- Log-Odds (Logit): The natural logarithm of the odds: ln(p / (1 - p)).

Logistic Regression asserts that the Log-Odds are linearly related to the predictors:
ln(p / (1 - p)) = beta_0 + beta_1 * X

In [ ]:
# Visualizing the relationship between Probability, Odds, and Log-Odds
probabilities = np.linspace(0.01, 0.99, 100)
odds = probabilities / (1 - probabilities)
log_odds = np.log(odds)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Probability vs Odds
axes[0].plot(probabilities, odds, color='purple', linewidth=2)
axes[0].set_title('Probability vs. Odds', fontsize=14)
axes[0].set_xlabel('Probability (p)', fontsize=12)
axes[0].set_ylabel('Odds (p / (1-p))', fontsize=12)
axes[0].grid(alpha=0.3)

# Plot 2: Probability vs Log-Odds
axes[1].plot(probabilities, log_odds, color='orange', linewidth=2)
axes[1].set_title('Probability vs. Log-Odds', fontsize=14)
axes[1].set_xlabel('Probability (p)', fontsize=12)
axes[1].set_ylabel('Log-Odds (ln(Odds))', fontsize=12)
axes[1].axhline(0, color='gray', linestyle='--')
axes[1].axvline(0.5, color='gray', linestyle='--')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("Notice that Log-Odds extends from negative to positive infinity.")
print("This is why we can safely fit a linear line (beta_0 + beta_1*X) to the Log-Odds!")

## 4. Core Concept 3: Maximum Likelihood Estimation (MLE)

Because the relationship between the log-odds and the predictors is nonlinear with respect to the raw outcome, we cannot use OLS to find the best-fitting line. We must use Maximum Likelihood Estimation (MLE).

We want to find the parameters (beta_0, beta_1) that make our observed binary outcomes most likely to have occurred. We do this by minimizing the Negative Log-Likelihood (also known as Log-Loss or Binary Cross-Entropy).

Log-Loss = - (1/N) * Sum [ y_i * ln(p_i) + (1 - y_i) * ln(1 - p_i) ]

In [ ]:
# Implementing Log-Loss and MLE from scratch
def negative_log_likelihood(weights, X, y):
    """Compute the negative log-likelihood (binary cross-entropy loss)."""
    # Compute linear combination
    z = weights[0] + weights[1] * X
    
    # Apply sigmoid
    p = sigmoid(z)
    
    # Clip probabilities to avoid log(0)
    p = np.clip(p, 1e-10, 1 - 1e-10)
    
    # Compute log-loss
    loss = -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))
    return loss

# Initial guess for [beta_0, beta_1]
initial_weights = np.array([0.0, 0.0])

# Use SciPy's minimize to find the optimal weights
result = minimize(negative_log_likelihood, initial_weights, args=(X_age.flatten(), y_readmit), method='BFGS')

mle_beta_0, mle_beta_1 = result.x

print("--- MLE Optimization Results (From Scratch) ---")
print(f"Optimization Success: {result.success}")
print(f"Optimal Intercept (beta_0): {mle_beta_0:.4f}")
print(f"Optimal Coefficient for Age (beta_1): {mle_beta_1:.4f}")

## 5. Implementation with Scikit-Learn

In practice, we use optimized libraries like Scikit-Learn to perform Logistic Regression. Let's build the model and compare it to our from-scratch MLE implementation.

In [ ]:
# Initialize and fit Logistic Regression model
# Note: penalty=None turns off regularization to match standard MLE
log_model = LogisticRegression(penalty=None)
log_model.fit(X_age, y_readmit)

sklearn_beta_0 = log_model.intercept_[0]
sklearn_beta_1 = log_model.coef_[0][0]

print("--- Scikit-Learn Results ---")
print(f"Intercept (beta_0): {sklearn_beta_0:.4f}")
print(f"Coefficient for Age (beta_1): {sklearn_beta_1:.4f}")
print("\nNotice how closely they match our custom MLE optimization!")

# Let's visualize the fitted Logistic curve
log_predictions = log_model.predict_proba(age_range)[:, 1] # Get probability of class 1

plt.figure(figsize=(10, 6))
plt.scatter(df_pharma['Age'], df_pharma['Readmitted'], alpha=0.5, color='blue', label='Actual Data')
plt.plot(age_range, log_predictions, color='green', linewidth=3, label='Logistic Regression Fit')

plt.axhline(0.5, color='red', linestyle='--', alpha=0.5, label='Decision Boundary (p=0.5)')

plt.title('Logistic Regression: Sigmoid Fit on Binary Data', fontsize=14)
plt.xlabel('Patient Age', fontsize=12)
plt.ylabel('Probability of Readmission', fontsize=12)
plt.legend(loc='upper left')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Interpreting the Coefficients (Odds Ratios)

In Linear Regression, beta_1 represents the absolute change in Y for a 1-unit change in X.
In Logistic Regression, beta_1 is the change in the Log-Odds. To make this interpretable to business stakeholders, we exponentiate the coefficient to get the Odds Ratio: exp(beta_1).

Instead of saying "an increase of 1 in Age increases log-odds by 0.12," we say "an increase of 1 in Age increases the odds of readmission by a factor of X."

In [ ]:
# Calculate Odds Ratio
odds_ratio = np.exp(sklearn_beta_1)
percentage_increase = (odds_ratio - 1) * 100

print("--- Pharmaceutical Business Interpretation ---")
print(f"Raw Coefficient (Log-Odds): {sklearn_beta_1:.4f}")
print(f"Odds Ratio (exp(beta)): {odds_ratio:.4f}")
print(f"\nInterpretation:")
print(f"For every 1 additional year of patient age, the odds of hospital readmission multiply by {odds_ratio:.2f}.")
print(f"This represents a {percentage_increase:.1f}% increase in the odds of readmission per year of age.")

## 7. Model Evaluation and Thresholding

Logistic regression outputs probabilities. To make a hard prediction (Yes/No), we must apply a threshold (usually 0.5).
Let's evaluate how well our model predicts readmissions.

In [ ]:
# Generate class predictions using standard 0.5 threshold
y_pred = log_model.predict(X_age)

# Compute Confusion Matrix
cm = confusion_matrix(y_readmit, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Not Readmitted', 'Readmitted'],
            yticklabels=['Not Readmitted', 'Readmitted'])
plt.title('Confusion Matrix', fontsize=14)
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.show()

print("\nClassification Report:")
print(classification_report(y_readmit, y_pred, target_names=['Not Readmitted', 'Readmitted']))

## 8. Expanding to Multiple Features (2D)

Let's add a second feature: Previous Hospital Visits. This allows us to visualize decision boundaries in 2D space.

In [ ]:
# Generate 2D Synthetic Data
n_samples_2d = 300
age_2d = np.random.uniform(30, 90, n_samples_2d)
prev_visits = np.random.poisson(lam=2, size=n_samples_2d)

# True relationship combining both features
z_2d = -8.0 + 0.1 * age_2d + 0.8 * prev_visits
prob_2d = sigmoid(z_2d)
readmitted_2d = np.random.binomial(1, prob_2d)

df_2d = pd.DataFrame({'Age': age_2d, 'Previous_Visits': prev_visits, 'Readmitted': readmitted_2d})

# Fit 2D Logistic Regression Model
X_2d = df_2d[['Age', 'Previous_Visits']]
y_2d = df_2d['Readmitted']

log_model_2d = LogisticRegression()
log_model_2d.fit(X_2d, y_2d)

print("Fitted Multi-Feature Logistic Regression Model.")
print(f"Coefficients: Age={log_model_2d.coef_[0][0]:.4f}, Prev_Visits={log_model_2d.coef_[0][1]:.4f}")

## 9. Visualization Gallery: Decision Boundaries

The decision boundary is the line where the predicted probability is exactly 0.5 (Log-Odds = 0).

In [ ]:
plt.figure(figsize=(10, 6))

# Scatter plot of actual data
sns.scatterplot(x='Age', y='Previous_Visits', hue='Readmitted', data=df_2d, 
                palette={0: 'blue', 1: 'red'}, alpha=0.7, s=60)

# Calculate the decision boundary line
# Equation: 0 = beta_0 + beta_1*Age + beta_2*Prev_Visits
# Prev_Visits = (-beta_0 - beta_1*Age) / beta_2
b0 = log_model_2d.intercept_[0]
b1 = log_model_2d.coef_[0][0]
b2 = log_model_2d.coef_[0][1]

x_boundary = np.array([30, 90])
y_boundary = (-b0 - b1 * x_boundary) / b2

plt.plot(x_boundary, y_boundary, color='black', linestyle='--', linewidth=2, label='Decision Boundary (p=0.5)')

plt.title('Logistic Regression Decision Boundary', fontsize=14)
plt.xlabel('Patient Age')
plt.ylabel('Previous Hospital Visits')
plt.ylim(0, df_2d['Previous_Visits'].max() + 1)
plt.legend()
plt.show()

print("Patients above/right of the line are predicted as Readmitted (1).")
print("Patients below/left of the line are predicted as Not Readmitted (0).")

## 10. Visualization Gallery: Probability Contours

A decision boundary is a hard cutoff, but Logistic Regression outputs continuous probabilities. Let's visualize the gradient of probability across our feature space.

In [ ]:
# Create a meshgrid to plot probability contours
xx, yy = np.meshgrid(np.linspace(30, 90, 100), np.linspace(0, 10, 100))
grid_data = np.c_[xx.ravel(), yy.ravel()]

# Predict probabilities for every point on the grid
probs = log_model_2d.predict_proba(grid_data)[:, 1].reshape(xx.shape)

plt.figure(figsize=(10, 6))

# Plot the probability contour
contour = plt.contourf(xx, yy, probs, alpha=0.6, cmap='coolwarm', levels=10)
plt.colorbar(contour, label='Predicted Probability of Readmission')

# Plot the actual data points
sns.scatterplot(x='Age', y='Previous_Visits', hue='Readmitted', data=df_2d, 
                palette={0: 'blue', 1: 'red'}, edgecolor='k', alpha=0.8)

plt.title('Probability Contours for Readmission', fontsize=14)
plt.xlabel('Patient Age')
plt.ylabel('Previous Hospital Visits')
plt.show()

## 11. Practice Exercise: Marketing Campaign Conversion

Scenario: You are analyzing a digital marketing campaign. You have data on the amount of Ad Spend per physician (in dollars) and whether they converted to prescribing your drug (0 = No, 1 = Yes).

Your task:
1. Fit a Logistic Regression model to this data.
2. Calculate the Odds Ratio for Ad Spend.
3. Print a business-friendly interpretation of what the Odds Ratio means.

In [ ]:
# Run this cell to generate your exercise data
n_doctors = 200
ad_spend = np.random.uniform(50, 500, n_doctors)

# True conversion probability logic
z_mkt = -4.0 + 0.015 * ad_spend
prob_mkt = sigmoid(z_mkt)
converted = np.random.binomial(1, prob_mkt)

df_marketing = pd.DataFrame({'Ad_Spend': ad_spend, 'Converted': converted})
print("Marketing Data Preview:")
print(df_marketing.head())

In [ ]:
# --- STUDENT WORKSPACE ---
# Write your code here:



# --- SOLUTION ---
X_mkt = df_marketing[['Ad_Spend']]
y_mkt = df_marketing['Converted']

# 1. Fit the model
mkt_model = LogisticRegression()
mkt_model.fit(X_mkt, y_mkt)
beta_spend = mkt_model.coef_[0][0]

# 2. Calculate Odds Ratio
odds_ratio_spend = np.exp(beta_spend)
pct_increase = (odds_ratio_spend - 1) * 100

# 3. Interpret
print("\n--- Exercise Solution ---")
print(f"Coefficient: {beta_spend:.4f}")
print(f"Odds Ratio: {odds_ratio_spend:.4f}")
print(f"Interpretation: For every additional $1 spent on ads, the odds of the physician converting increase by {pct_increase:.2f}%.")

## 12. Advanced Evaluation: ROC Curve and AUC

Instead of just looking at the hard 0.5 threshold, we evaluate classification models across all possible thresholds using the Receiver Operating Characteristic (ROC) Curve. The Area Under the Curve (AUC) is a standard metric for model performance (1.0 is perfect, 0.5 is random guessing).

In [ ]:
# Calculate probabilities for the 2D Patient model
y_prob_2d = log_model_2d.predict_proba(X_2d)[:, 1]

# Compute False Positive Rate, True Positive Rate, and Thresholds
fpr, tpr, thresholds = roc_curve(y_2d, y_prob_2d)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Guessing')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=12)
plt.ylabel('True Positive Rate (Sensitivity)', fontsize=12)
plt.title('Receiver Operating Characteristic (ROC) Curve', fontsize=14)
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()

print(f"An AUC of {roc_auc:.2f} indicates the model is highly effective at distinguishing between readmitted and non-readmitted patients.")

## 13. Summary and Key Takeaways

- The Linear Probability Model Fails: Linear regression cannot constrain predictions to [0, 1] and suffers from heteroskedasticity when applied to binary data.
- The Logit Solution: We use the Sigmoid function to squash linear combinations into valid probabilities.
- Odds & Log-Odds: Logistic regression models the Log-Odds linearly. We exponentiate coefficients to get Odds Ratios, which are highly interpretable for business stakeholders.
- Maximum Likelihood Estimation: We fit the model by minimizing the Negative Log-Likelihood (Binary Cross-Entropy), finding the parameters that make our observed data most probable.
- Evaluation: Use Confusion Matrices for thresholded accuracy, and ROC/AUC curves for holistic model evaluation across all probability thresholds.